# Week 11 Pandas and DuckDB Examples

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST2412_Data_Security_Privacy_Ethics/blob/main/week_11/week11_pandas_duckdb_examples.ipynb)

Companion notebook for Week 11.
This notebook mirrors the Day 1 and Day 2 analytics logic used in the Week 11 labs.

What it includes:
- `pandas` examples for Day 1 and Day 2
- `DuckDB` SQL examples for Day 1 and Day 2
- GitHub raw URLs so the notebook can pull the current Week 11 CSVs from the repository

Files used:
- `week_11/data/day1_auth_events.csv`
- `week_11/data/day1_user_context.csv`
- `week_11/data/day2_alert_queue.csv`
- `week_11/data/day2_asset_context.csv`


## How to read this notebook if you are new to Python

This notebook is written for beginners.
A few ideas will appear over and over:

- a **DataFrame** is like a table in Excel or Google Sheets
- a **column** is one field, such as `src_ip` or `user_id`
- a **row** is one recorded event or one alert
- `pandas` is a Python library for working with tables
- `DuckDB` lets us run SQL directly on CSV files

When you see code in this notebook, read it in this order:
1. what file is being loaded
2. what rows are being filtered
3. what columns are being counted or joined
4. what the final table is trying to show

You do not need to memorize every line.
The goal is to understand what kind of question the code is answering.


In [ ]:
# If needed, uncomment this cell and run it once in Colab.
# This installs the two Python packages used in the notebook:
# - pandas: for table/dataframe work in Python
# - duckdb: for SQL queries directly on CSV files
# %pip install pandas duckdb


In [ ]:
# Import the libraries we need.
# `import ... as ...` gives each library a short nickname.
# `pd` is the standard short name for pandas.
import pandas as pd
import duckdb

# All four CSVs live in the GitHub repository.
# We keep one base URL so we do not have to repeat the long path four times.
BASE_URL = 'https://github.com/lolusername/CST2412_Data_Security_Privacy_Ethics/raw/main/week_11/data'

# Build the full URL for each CSV.
# The f"..." syntax is called an f-string.
# It lets us insert the BASE_URL into a longer string.
DAY1_EVENTS = f"{BASE_URL}/day1_auth_events.csv"
DAY1_USERS = f"{BASE_URL}/day1_user_context.csv"
DAY2_ALERTS = f"{BASE_URL}/day2_alert_queue.csv"
DAY2_ASSETS = f"{BASE_URL}/day2_asset_context.csv"

print('Base URL:', BASE_URL)
print('Day 1 events:', DAY1_EVENTS)
print('Day 2 alerts:', DAY2_ALERTS)


## Day 1 with Pandas

Goal:
- inspect authentication activity
- count failures by source IP
- identify the likely password-spray source
- enrich the suspicious activity with account context

### New pandas functions in this section
- `pd.read_csv(...)`: loads a CSV into a DataFrame
- `.query(...)`: filters rows using a readable condition
- `.groupby(...)`: groups rows that share the same value
- `.agg(...)`: calculates summaries like counts
- `.sort_values(...)`: sorts rows by one or more columns
- `.merge(...)`: joins two tables together
- `.loc[...]`: selects rows and columns explicitly


In [ ]:
# Load the Day 1 CSV files into pandas DataFrames.
# Think of each DataFrame as an in-memory table.
# `parse_dates=['timestamp']` tells pandas to treat the timestamp column
# as a real date/time value instead of plain text.
events = pd.read_csv(DAY1_EVENTS, parse_dates=['timestamp'])
users = pd.read_csv(DAY1_USERS)

# `len(events)` returns the number of rows in the events table.
print('Total rows:', len(events))

# `events['event_type']` means: give me the event_type column.
# `.value_counts()` counts how many times each distinct value appears.
print('Event types:')
print(events['event_type'].value_counts())

# Show the first few rows so students can see the raw structure.
# `.head()` returns the first 5 rows by default.
events.head()


In [ ]:
# Keep only rows where the event type is login_failed.
# `.query(...)` is a readable way to filter a table.
failed = events.query("event_type == 'login_failed'")

# `.groupby('src_ip')` means: collect together all rows that share the same source IP.
# `.agg(...)` lets us calculate more than one summary at once.
# - count: how many failed rows came from that IP
# - nunique: how many DISTINCT user IDs were targeted by that IP
failed_by_ip = (
    failed.groupby('src_ip')
    .agg(
        failed_count=('event_id', 'count'),
        targeted_users=('user_id', 'nunique')
    )
    .sort_values('failed_count', ascending=False)
)

# Show the summary table.
# The most suspicious IPs often float to the top when we sort by failed_count descending.
failed_by_ip


In [ ]:
# The DataFrame index now stores the src_ip values.
# `index[0]` means: give me the first row label in the sorted result.
# Because we sorted from highest failed_count to lowest, this is our most suspicious source IP.
spray_ip = failed_by_ip.index[0]
print('Likely suspicious source IP:', spray_ip)

# `.loc[row_condition, [columns...]]` is one of the most important pandas patterns.
# It means:
# 1. keep only rows where src_ip matches spray_ip
# 2. only show these specific columns
# 3. sort the result by timestamp so we can read the sequence in time order
failed.loc[
    failed['src_ip'] == spray_ip,
    ['timestamp', 'user_id', 'device', 'city']
].sort_values('timestamp')


In [ ]:
# `.merge(...)` joins the events table with the user-context table.
# `on='user_id'` means match rows using the user_id column.
# `how='left'` means keep every event row, even if some user context were missing.
enriched_events = events.merge(users, on='user_id', how='left')

# This filter keeps rows that are either:
# - from the suspicious IP, OR
# - for the specific finance admin account we want to inspect closely
# The `|` symbol means OR.
enriched_view = enriched_events.loc[
    (enriched_events['src_ip'] == spray_ip) | (enriched_events['user_id'] == 'u_admin_fin'),
    ['timestamp', 'src_ip', 'user_id', 'event_type', 'outcome', 'privileged_account', 'sensitive_account', 'department', 'role_title']
].sort_values(['timestamp', 'user_id'])

# Notice what enrichment does for us:
# the raw event data now sits next to account context like privilege and sensitivity.
enriched_view


In [ ]:
# Build a focused sequence just for the finance admin account.
# This is a good way to tell a smaller, clearer story inside a larger dataset.
u_admin_fin_sequence = enriched_events.loc[
    enriched_events['user_id'] == 'u_admin_fin',
    ['timestamp', 'src_ip', 'event_type', 'outcome', 'factor', 'privileged_account', 'sensitive_account']
].sort_values('timestamp')

u_admin_fin_sequence


## Day 1 with DuckDB SQL

DuckDB is useful because it lets us run SQL directly on CSV files.
That means students can compare the Python version and the SQL version of the same logic.

### SQL ideas to notice
- `SELECT`: choose which columns to return
- `FROM`: choose the data source
- `WHERE`: filter rows
- `GROUP BY`: group rows before counting
- `COUNT(...)`: summarize rows
- `LEFT JOIN`: attach context from another table
- `ORDER BY`: sort the final result


In [ ]:
# Create a DuckDB connection inside the notebook.
# This does not create a separate server.
# It just gives us a SQL engine we can talk to.
con = duckdb.connect()

# This SQL query does the same basic job as the pandas groupby example above.
# It reads the CSV directly from GitHub and counts failed logins per source IP.
day1_count_query = f"""
SELECT
    src_ip,
    COUNT(*) AS failed_count,
    COUNT(DISTINCT user_id) AS targeted_users
FROM read_csv_auto('{DAY1_EVENTS}')
WHERE event_type = 'login_failed'
GROUP BY src_ip
ORDER BY failed_count DESC;
"""

# `con.sql(...)` runs the query.
# `.df()` converts the result back into a pandas DataFrame so it displays nicely in the notebook.
con.sql(day1_count_query).df()


In [ ]:
# This second SQL query shows enrichment.
# We join the events CSV to the user-context CSV on user_id.
# That reproduces the same idea as pandas `.merge(...)`.
day1_enrich_query = f"""
SELECT
    e.timestamp,
    e.src_ip,
    e.user_id,
    e.event_type,
    e.outcome,
    u.department,
    u.role_title,
    u.privileged_account,
    u.sensitive_account
FROM read_csv_auto('{DAY1_EVENTS}') AS e
LEFT JOIN read_csv_auto('{DAY1_USERS}') AS u
    ON e.user_id = u.user_id
WHERE e.src_ip = '198.51.100.77'
ORDER BY e.timestamp;
"""

con.sql(day1_enrich_query).df()


## Day 2 with Pandas

Goal:
- join the alert queue to asset context
- compute the lab's scoring rubric
- rank alerts
- inspect the top escalation candidate

### New pandas functions in this section
- `.map(...)`: replaces text values using a dictionary
- `.astype(int)`: converts True/False values into 1/0 values for scoring
- `.iloc[...]`: selects a row by numeric position
- `.isin(...)`: checks whether a value appears in a given list


In [ ]:
# Load the Day 2 alert queue and asset context.
alerts = pd.read_csv(DAY2_ALERTS)
assets = pd.read_csv(DAY2_ASSETS)

# Build dictionaries that convert text labels into numeric scores.
# Example: "high" becomes 3, "medium" becomes 2, and "low" becomes 1.
severity_map = {'high': 3, 'medium': 2, 'low': 1}
confidence_map = {'high': 3, 'medium': 2, 'low': 1}
criticality_map = {'high': 3, 'medium': 2, 'low': 1}

# Join the alert data to the asset data.
# This is the Day 2 version of enrichment.
merged = alerts.merge(assets, on='asset_id', how='left')

# Create a numeric score column using the rubric from the lab.
# `.map(...)` converts text labels like "high" into numbers.
# `(merged['internet_exposed'] == 'yes')` creates True/False values.
# `.astype(int)` turns True into 1 and False into 0 so we can add them.
merged['score'] = (
    merged['severity'].map(severity_map)
    + merged['confidence'].map(confidence_map)
    + merged['asset_criticality'].map(criticality_map)
    + (merged['internet_exposed'] == 'yes').astype(int)
    + (merged['sensitive_data'] == 'yes').astype(int)
)

# Show the ranked alert table.
# We sort by score from highest to lowest.
# If two alerts have the same score, we break the tie alphabetically by alert_id.
merged[
    ['alert_id', 'alert_type', 'asset_name', 'severity', 'confidence', 'asset_criticality', 'internet_exposed', 'sensitive_data', 'score']
].sort_values(['score', 'alert_id'], ascending=[False, True])


In [ ]:
# `.iloc[0]` means: give me the first row by numeric position.
# Because the table is sorted from highest score to lowest,
# this returns the top-ranked alert.
top_alert = merged.sort_values(['score', 'alert_id'], ascending=[False, True]).iloc[0]
top_alert


In [ ]:
# `.isin([...])` checks whether each alert_id appears in this list.
# That lets us pull a smaller teaching subset from the larger queue.
merged.loc[
    merged['alert_id'].isin(['A1006', 'A1001', 'A1002', 'A1003']),
    ['alert_id', 'alert_type', 'asset_name', 'notes', 'score']
].sort_values(['score', 'alert_id'], ascending=[False, True])


## Day 2 with DuckDB SQL

This section reproduces the same scoring logic in SQL.
The main idea is the same as the pandas version:
- join alerts to assets
- convert text labels into numeric values
- compute a total score
- sort the results


In [ ]:
# This query creates a temporary result called `scored` using a CTE.
# CTE stands for Common Table Expression.
# You can think of it as a named intermediate table.
day2_score_query = f"""
WITH scored AS (
    SELECT
        a.alert_id,
        a.alert_type,
        a.severity,
        a.confidence,
        x.asset_name,
        x.asset_criticality,
        x.internet_exposed,
        x.sensitive_data,
        a.notes,
        CASE a.severity
            WHEN 'high' THEN 3
            WHEN 'medium' THEN 2
            ELSE 1
        END
        + CASE a.confidence
            WHEN 'high' THEN 3
            WHEN 'medium' THEN 2
            ELSE 1
        END
        + CASE x.asset_criticality
            WHEN 'high' THEN 3
            WHEN 'medium' THEN 2
            ELSE 1
        END
        + CASE WHEN x.internet_exposed = 'yes' THEN 1 ELSE 0 END
        + CASE WHEN x.sensitive_data = 'yes' THEN 1 ELSE 0 END
        AS total_score
    FROM read_csv_auto('{DAY2_ALERTS}') AS a
    LEFT JOIN read_csv_auto('{DAY2_ASSETS}') AS x
        ON a.asset_id = x.asset_id
)
SELECT *
FROM scored
ORDER BY total_score DESC, alert_id;
"""

con.sql(day2_score_query).df()


## Suggested teaching use

Use this notebook if you want to:
- show students that the CSV logic can be expressed in code
- demonstrate how enrichment works in `pandas` and SQL
- model the difference between raw events and context-enriched analysis
- preview how analyst logic transfers into data tooling

Teaching advice:
- For beginners, narrate the code line by line the first time.
- Keep bringing students back to the security question each cell is answering.
- Remind them that the code is just another way to express the same reasoning they already practiced in the CSV labs.
